# 06 — Amélioration du modèle avec météo 2025

**Objectif** : comparer XGBoost baseline (sans météo) vs XGBoost enrichi (avec météo)
sur la même période 2025 pour évaluer l'impact réel de la météo.

**Données**
- Pollution 2024 (baseline) et 2025 (enrichi)
- Météo daily 2025 → interpolée à horaire
- Split : train 2025 H1, val 2025 H2, test 2026

**Question clé** : est-ce que la météo améliore vraiment la prédiction de pollution ?
Si oui de combien (%) ? Les variables météo sont-elles importantes ?


In [ ]:
# Setup
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error

from src.features.pollution_features import build_dataset, temporal_split, SEUILS_LEGAUX, SEGMENTS
from src.features.meteo_preprocessing import prepare_meteo_2025, merge_pollution_meteo, extract_meteo_features_for_model
from src.models.pollution_predictor import PollutionPredictor, evaluate_predictions

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (13, 5)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

POLLUTANT = "NO2"
THRESHOLD = SEUILS_LEGAUX[POLLUTANT]
DATA_DIR = project_root / "data"

print("✓ Imports OK")


## Étape 1 : Charger et préparer la météo 2025


In [ ]:
# Charger météo daily 2025
meteo_path = DATA_DIR / "meteo_2025_daily.csv"
print(f"Chemin météo : {meteo_path}")
print(f"Existe : {meteo_path.exists()}")

if meteo_path.exists():
    # Préparer météo hourly
    meteo_hourly = prepare_meteo_2025(meteo_path)
    print(f"\n✓ Météo hourly : {len(meteo_hourly)} lignes")
    print(f"Colonnes disponibles : {meteo_hourly.columns.tolist()}")
    print(f"\nAperçu :\n{meteo_hourly.head(5)}")
else:
    print("\n⚠️ Fichier météo non trouvé. Place 'meteo_2025_daily.csv' dans data/")
    print("Les étapes suivantes seront simulées.")


## Étape 2 : Construire dataset pollution 2025 + météo


In [ ]:
# Charger pollution pour 2025
df_pollution = build_dataset(
    air_quality_path=DATA_DIR / "processed/air_quality_peripherique_prepared.csv",
    holidays_path=DATA_DIR / "jours_feries_metropole (1).csv",
    weather_path=None,
    pollutants=[POLLUTANT],
)

# Garder seulement 2025
df_2025 = df_pollution[df_pollution["time"] >= "2025-01-01"].copy()
print(f"Pollution 2025 : {len(df_2025)} lignes")
print(f"Période : {df_2025['time'].min()} -> {df_2025['time'].max()}")

# Merger avec météo si disponible
if meteo_path.exists():
    df_2025_meteo = merge_pollution_meteo(
        DATA_DIR / "processed/air_quality_peripherique_prepared.csv",
        meteo_hourly,
        time_col="time"
    )
    # Garder seulement 2025
    df_2025_meteo = df_2025_meteo[df_2025_meteo["time"] >= "2025-01-01"].copy()
    print(f"\n✓ Dataset pollution + météo 2025 : {len(df_2025_meteo)} lignes")
    print(f"NaN par colonne météo:\n{df_2025_meteo.isna().sum()}")
else:
    # Fallback : utiliser pollution 2025 seule (simulation)
    df_2025_meteo = df_2025.copy()
    print("⚠️ Météo non disponible - simulation avec colonnes factices")
    meteo_cols = ["temperature", "humidity", "precipitation", "pressure", "cloudcover"]
    for col in meteo_cols:
        df_2025_meteo[col] = np.random.normal(15, 5, len(df_2025_meteo))


## Étape 3 : Split temporel

On utilise 2025 complet pour l'enrichissement (plus de données = meilleur apprentissage).
Pour la validation, on utilise aussi 2026 si dispo.


In [ ]:
# Split sur 2025
train_2025, val_2025, test_2025 = temporal_split(
    df_2025_meteo,
    train_end="2025-06-30 23:00:00",
    val_end="2025-10-31 23:00:00",
)

# Nettoyer les lags NaN
lag_cols = [c for c in train_2025.columns if c.startswith("lag_") or c.startswith("rolling_")]
train_2025_c = train_2025.dropna(subset=lag_cols)
val_2025_c = val_2025.dropna(subset=lag_cols)
test_2025_c = test_2025.dropna(subset=lag_cols)

print(f"Train 2025 : {len(train_2025_c):,} lignes | {train_2025_c['time'].min().date()} -> {train_2025_c['time'].max().date()}")
print(f"Val 2025   : {len(val_2025_c):,} lignes | {val_2025_c['time'].min().date()} -> {val_2025_c['time'].max().date()}")
print(f"Test 2025  : {len(test_2025_c):,} lignes | {test_2025_c['time'].min().date()} -> {test_2025_c['time'].max().date()}")


## Étape 4 : Modèle baseline (sans météo, entraîné 2024)

Pour la comparaison équitable, on prend le modèle du notebook 04 entraîné sur 2024
et on l'évalue sur 2025.


In [ ]:
# Charger pollution 2024 pour baseline
df_2024 = df_pollution[df_pollution["time"] < "2025-01-01"].copy()
lag_cols = [c for c in df_2024.columns if c.startswith("lag_") or c.startswith("rolling_")]
df_2024_c = df_2024.dropna(subset=lag_cols)

print(f"Pollution 2024 : {len(df_2024_c):,} lignes")

# Entraîner baseline sur 2024
model_baseline = PollutionPredictor(pollutant=POLLUTANT, n_estimators=500, max_depth=6, random_state=42)
model_baseline.fit(df_2024_c)
print(f"✓ Baseline entraîné sur 2024")

# Évaluer sur val 2025
metrics_baseline = model_baseline.evaluate(val_2025_c, threshold=THRESHOLD)
print(f"\n=== Baseline (2024 → 2025) ===")
print(f"MAE  : {metrics_baseline['MAE']:.3f} µg/m³")
print(f"RMSE : {metrics_baseline['RMSE']:.3f} µg/m³")
print(f"Recall (dépassements) : {metrics_baseline['recall_exceed']:.1%}")


## Étape 5 : Modèle enrichi (avec météo, entraîné 2025)

On entraîne sur 2025 qui a la météo complète. Le modèle apprendra la relation
pollution ↔ météo explicitement.


In [ ]:
# Entraîner modèle enrichi sur 2025 (avec météo)
model_meteo = PollutionPredictor(pollutant=POLLUTANT, n_estimators=500, max_depth=6, random_state=42)
model_meteo.fit(train_2025_c)
print(f"✓ Modèle enrichi entraîné sur 2025 (avec météo)")

# Évaluer sur val 2025
metrics_meteo = model_meteo.evaluate(val_2025_c, threshold=THRESHOLD)
print(f"\n=== Modèle enrichi (2025 avec météo) ===")
print(f"MAE  : {metrics_meteo['MAE']:.3f} µg/m³")
print(f"RMSE : {metrics_meteo['RMSE']:.3f} µg/m³")
print(f"Recall (dépassements) : {metrics_meteo['recall_exceed']:.1%}")

# Amélioration
improvement_mae = (metrics_baseline['MAE'] - metrics_meteo['MAE']) / metrics_baseline['MAE'] * 100
improvement_rmse = (metrics_baseline['RMSE'] - metrics_meteo['RMSE']) / metrics_baseline['RMSE'] * 100

print(f"\n=== Amélioration ===")
print(f"MAE : {improvement_mae:+.1f}%")
print(f"RMSE : {improvement_rmse:+.1f}%")

if improvement_mae > 10:
    print("✅ Météo apporte une amélioration SIGNIFICATIVE")
elif improvement_mae > 5:
    print("✓ Météo apporte une amélioration MODÉRÉE")
else:
    print("⚠️ Météo apporte peu d'amélioration (contexte temp + lags suffisent)")


## Étape 6 : Feature importance - d'où vient l'amélioration ?

On compare l'importance des features météo vs features temporelles dans le modèle enrichi.


In [ ]:
# Feature importance du modèle enrichi
imp_meteo = model_meteo.feature_importance(top=20)

print("Top 20 features (modèle enrichi avec météo) :")
print(imp_meteo)

# Séparer météo vs temporel
meteo_features = [f for f in imp_meteo['feature'] if any(m in f.lower() for m in 
                   ['temp', 'humid', 'precip', 'pressure', 'cloud', 'wind', 'rainy', 'hot', 'cold', 'cloudy'])]
temporal_features = [f for f in imp_meteo['feature'] if f not in meteo_features]

imp_meteo['type'] = imp_meteo['feature'].apply(
    lambda f: 'Météo' if f in meteo_features else 'Temporel/Lag'
)

fig, ax = plt.subplots(figsize=(11, 7))
colors = {'Météo': '#E76F51', 'Temporel/Lag': '#2A9D8F'}
for ftype in ['Temporel/Lag', 'Météo']:
    subset = imp_meteo[imp_meteo['type'] == ftype].head(10)
    ax.barh(subset['feature'], subset['importance'], label=ftype, color=colors[ftype], alpha=0.85)

ax.set_xlabel("Importance")
ax.set_title(f"Feature importance — Modèle enrichi avec météo")
ax.legend()
plt.tight_layout()
plt.show()

# Résumé
imp_by_type = imp_meteo.groupby('type')['importance'].sum()
print(f"\nImportance totale par type :")
print(imp_by_type)
print(f"Météo / Total : {imp_by_type.get('Météo', 0) / imp_by_type.sum() * 100:.1f}%")


## Étape 7 : Visualisation côte à côte


In [ ]:
# Tableau récapitulatif
comparison = pd.DataFrame([
    {
        'Modèle': 'Baseline (2024)',
        'Entraîné sur': '2024',
        'Météo': 'Non',
        'MAE': metrics_baseline['MAE'],
        'RMSE': metrics_baseline['RMSE'],
        'Recall': metrics_baseline['recall_exceed'],
        'Precision': metrics_baseline['precision_exceed'],
    },
    {
        'Modèle': 'Enrichi (2025)',
        'Entraîné sur': '2025',
        'Météo': 'Oui',
        'MAE': metrics_meteo['MAE'],
        'RMSE': metrics_meteo['RMSE'],
        'Recall': metrics_meteo['recall_exceed'],
        'Precision': metrics_meteo['precision_exceed'],
    }
])

print("\n=== Comparaison complète ===")
print(comparison.to_string(index=False))

# Graphiques
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# MAE
axes[0,0].bar(['Baseline', 'Enrichi'], [metrics_baseline['MAE'], metrics_meteo['MAE']], 
              color=['#264653', '#2A9D8F'])
axes[0,0].set_title('MAE (µg/m³) — plus bas = mieux')
axes[0,0].set_ylabel('MAE')

# RMSE
axes[0,1].bar(['Baseline', 'Enrichi'], [metrics_baseline['RMSE'], metrics_meteo['RMSE']], 
              color=['#264653', '#2A9D8F'])
axes[0,1].set_title('RMSE (µg/m³)')
axes[0,1].set_ylabel('RMSE')

# Recall dépassement
axes[1,0].bar(['Baseline', 'Enrichi'], 
              [metrics_baseline['recall_exceed'], metrics_meteo['recall_exceed']], 
              color=['#264653', '#E76F51'])
axes[1,0].set_title(f'Recall détection dépassements >{THRESHOLD} µg/m³')
axes[1,0].set_ylabel('Recall')
axes[1,0].set_ylim([0, 1])

# Precision dépassement
axes[1,1].bar(['Baseline', 'Enrichi'], 
              [metrics_baseline['precision_exceed'], metrics_meteo['precision_exceed']], 
              color=['#264653', '#E76F51'])
axes[1,1].set_title(f'Precision détection dépassements >{THRESHOLD} µg/m³')
axes[1,1].set_ylabel('Precision')
axes[1,1].set_ylim([0, 1])

plt.suptitle('Baseline vs Modèle enrichi avec météo', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()


## Étape 8 : Analyse des erreurs par conditions météo

On regarde si le modèle enrichi réduit les erreurs dans certaines conditions.


In [ ]:
# Ajouter prédictions et erreurs
val_2025_c_pred = val_2025_c.copy()
val_2025_c_pred['pred_baseline'] = model_baseline.predict(val_2025_c)
val_2025_c_pred['pred_meteo'] = model_meteo.predict(val_2025_c)
val_2025_c_pred['err_baseline'] = np.abs(val_2025_c_pred['pred_baseline'] - val_2025_c_pred['value'])
val_2025_c_pred['err_meteo'] = np.abs(val_2025_c_pred['pred_meteo'] - val_2025_c_pred['value'])

# Erreur moyenne par contexte
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Par heure
err_by_hour = pd.DataFrame({
    'Baseline': val_2025_c_pred.groupby('hour')['err_baseline'].mean(),
    'Enrichi': val_2025_c_pred.groupby('hour')['err_meteo'].mean(),
})
err_by_hour.plot(ax=axes[0], marker='o')
axes[0].set_title('MAE par heure du jour')
axes[0].set_xlabel('Heure')
axes[0].set_ylabel('MAE (µg/m³)')
axes[0].legend()

# Par jour de semaine
day_names = ['Lun', 'Mar', 'Mer', 'Jeu', 'Ven', 'Sam', 'Dim']
err_by_dow = pd.DataFrame({
    'Baseline': val_2025_c_pred.groupby('day_of_week')['err_baseline'].mean(),
    'Enrichi': val_2025_c_pred.groupby('day_of_week')['err_meteo'].mean(),
}, index=day_names)
err_by_dow.plot(ax=axes[1], marker='s')
axes[1].set_title('MAE par jour de semaine')
axes[1].set_ylabel('MAE (µg/m³)')
axes[1].legend()

# Par température (si dispo)
if 'temperature' in val_2025_c_pred.columns:
    val_2025_c_pred['temp_bin'] = pd.cut(val_2025_c_pred['temperature'], bins=5, labels=['Froid', 'Frais', 'Moyen', 'Chaud', 'Très chaud'])
    err_by_temp = pd.DataFrame({
        'Baseline': val_2025_c_pred.groupby('temp_bin', observed=True)['err_baseline'].mean(),
        'Enrichi': val_2025_c_pred.groupby('temp_bin', observed=True)['err_meteo'].mean(),
    })
    err_by_temp.plot(ax=axes[2], marker='^')
    axes[2].set_title('MAE par régime thermique')
    axes[2].set_ylabel('MAE (µg/m³)')
    axes[2].legend()
else:
    axes[2].text(0.5, 0.5, 'Colonne température non disponible', ha='center', va='center')

plt.tight_layout()
plt.show()


## Étape 9 : Synthèse et recommandations

### Conclusion


In [ ]:
# Synthèse en texte
print("=" * 60)
print("SYNTHÈSE : IMPACT DE LA MÉTÉO")
print("=" * 60)

if improvement_mae > 10:
    conclusion = "FAVORABLE : la météo apporte une amélioration SIGNIFICATIVE"
    recommendation = "Inclure météo dans le modèle de production"
    impact = "HIGH"
elif improvement_mae > 5:
    conclusion = "MODÉRÉ : la météo apporte une amélioration notable"
    recommendation = "Inclure météo, mais lags + contexte suffisent déjà"
    impact = "MEDIUM"
else:
    conclusion = "MINIMAL : la météo n'apporte pas d'amélioration majeure"
    recommendation = "Conserver modèle simple (baseline sans météo)"
    impact = "LOW"

print(f"\nConclusion : {conclusion}")
print(f"Recommandation : {recommendation}")
print(f"Importance impact : {impact}")

print(f"\nChiffres clés :")
print(f"  • MAE baseline 2024→2025 : {metrics_baseline['MAE']:.3f} µg/m³")
print(f"  • MAE enrichi 2025 : {metrics_meteo['MAE']:.3f} µg/m³")
print(f"  • Amélioration : {improvement_mae:+.1f}%")
print(f"\n  • Recall (détection dépassements)")
print(f"    - Baseline : {metrics_baseline['recall_exceed']:.1%}")
print(f"    - Enrichi : {metrics_meteo['recall_exceed']:.1%}")

print(f"\nFeatures météo dans top 20 : {len(meteo_features)} / 20")
print(f"Importance météo / Total : {imp_by_type.get('Météo', 0) / imp_by_type.sum() * 100:.1f}%")

print("\n" + "=" * 60)


## Étape 10 : Sauvegarder les modèles

Les deux modèles sont sauvegardés pour la production.


In [ ]:
# Sauvegarder
models_dir = project_root / "models"
models_dir.mkdir(exist_ok=True)

model_baseline.save(models_dir / "pollution_xgb_no2_baseline_2024.pkl")
model_meteo.save(models_dir / "pollution_xgb_no2_meteo_2025.pkl")

print(f"✓ Modèles sauvegardés :")
print(f"  - {models_dir / 'pollution_xgb_no2_baseline_2024.pkl'}")
print(f"  - {models_dir / 'pollution_xgb_no2_meteo_2025.pkl'}")
